# Stage 06 — Fixed-Budget Review and Reliability Evaluation

## Stage Overview
This stage consumes immutable Stage 05 predictions only and evaluates H4 under the locked 5%, 10%, and 20% review budgets, with 10% primary.

## Scientific Objective
Positive-risk ranking and uncertainty/error-review utility remain distinct operational tasks. Scalar temperature scaling is not claimed to improve a pure monotonic top-k ordering.


## Input Contract

### Required Input
The frozen Swarabyanjan dataset may remain mounted but is not read by Stage 06.

### Runtime
Kaggle Notebook.

### Accelerator
**CPU sufficient.** No model training or transformer inference occurs.

### Pipeline Position
03 / 04 / 05 / 06

### Upstream Requirement
Mount the **successful Stage-05 notebook output directory itself**. No separately packaged H3-finalization ZIP is required.

The mounted Stage-05 output must contain:
- `STAGE_STATUS.json`
- `STAGE05_ARTIFACT_MANIFEST.json`
- `STAGE05_SHA256SUMS.txt`
- `LOCKED_SOURCE_TEST_PREDICTIONS.parquet`
- `BANGLACLICK_PLATFORM_PREDICTIONS.parquet`
- `BAITBUSTER_HUMAN_PREDICTIONS.parquet`

Stage 06 validates that Stage 05 is `PASS`, verifies the Stage-05 manifest payload seal and checksum contract, and verifies the exact physical prediction hashes before H4 analysis.

### H3 boundary
H4 does **not** consume H3 estimates, H3 p-values, H3 summaries, or any repaired H3 ZIP. Stage-05 H3 finalization is scientifically upstream but is not an input dependency of the H4 computation. This prevents stale H3 summary artifacts from affecting Stage 06.

### Generated Outputs
`/kaggle/working/03_results/06_fixed_budget_reliability_evaluation/`, including H4 results with explicit estimability records, fixed-budget tables, risk-coverage data, bootstrap runtime diagnostics, figures/tables, final statistical summary, reproducibility manifest, and hashes.

### Manual Uploads
No model files, external datasets, thresholds, H3 ZIP, or intermediate model outputs are required beyond the mounted successful Stage-05 output.

## Runtime Requirements
- Underlying Stage-05 prediction run must be `PASS`.
- CPU is sufficient; GPU and Internet are not required.
- No manual path, threshold, budget, or target-data configuration is required.


## Environment Validation
Only the existing Kaggle data-analysis stack is used. Missing plotting support may be installed without upgrading the compiled scientific stack; an import probe runs before analysis.


In [1]:
# Environment preflight — standard library only.
# The compiled Kaggle scientific stack is preserved. Only missing application-level packages may be installed.
from pathlib import Path
import importlib.metadata as _md
import importlib.util as _util
import json as _json
import os as _os
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
import textwrap as _textwrap

STAGE_ENVIRONMENT_ID = "SWARABYANJAN-KAGGLE-ENV-v2.0.0"
CORE_DISTRIBUTIONS = {
    "numpy":"numpy",
    "pandas":"pandas",
    "pyarrow":"pyarrow",
}
APPLICATION_REQUIREMENTS = [('matplotlib', 'matplotlib', 'matplotlib>=3.7,<4')]

def _dist_version(name):
    try:
        return _md.version(name)
    except _md.PackageNotFoundError:
        return None

def _module_present(name):
    try:
        return _util.find_spec(name) is not None
    except Exception:
        return False

initial_versions = {name:_dist_version(name) for name in list(CORE_DISTRIBUTIONS)+[x[0] for x in APPLICATION_REQUIREMENTS]}
missing_core = [dist for dist,module in CORE_DISTRIBUTIONS.items() if not _module_present(module)]
if missing_core:
    raise RuntimeError(
        "WHAT FAILED: the standard Kaggle compiled scientific stack is incomplete. "
        f"Missing core distributions/modules: {missing_core}. "
        "WHY: installing or replacing NumPy/SciPy/pandas/scikit-learn/pyarrow/torch inside this notebook can create ABI conflicts. "
        "SAFE TO CONTINUE: NO. Start a fresh standard Kaggle Python runtime; do not upgrade the scientific stack."
    )

missing_apps = [(dist,module,spec) for dist,module,spec in APPLICATION_REQUIREMENTS if not _module_present(module)]
installed_now = []
if missing_apps:
    # Pin every existing compiled/core package so the dependency resolver cannot replace Kaggle's ABI-compatible stack.
    constraints = []
    for dist in CORE_DISTRIBUTIONS:
        v = _dist_version(dist)
        if v:
            constraints.append(f"{dist}=={v}")
    cfile = Path(_tempfile.gettempdir()) / f"swarabyanjan_06_constraints.txt"
    cfile.write_text("\n".join(constraints)+"\n", encoding="utf-8")
    specs = [spec for _,_,spec in missing_apps]
    cmd = [
        _sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--no-input", "--prefer-binary",
        "--constraint", str(cfile), *specs
    ]
    proc = _subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            "WHAT FAILED: installation of missing application-level dependencies failed. "
            f"Requested: {specs}. "
            "WHY: pip could not satisfy them without changing the pinned Kaggle scientific stack or network/package access failed. "
            "SAFE TO CONTINUE: NO.\n"
            f"pip stdout:\n{proc.stdout[-4000:]}\n"
            f"pip stderr:\n{proc.stderr[-4000:]}"
        )
    installed_now = specs

# Import/API validation occurs in a fresh subprocess before compiled packages are imported into this notebook kernel.
_probe = r"""
import json
import numpy, pandas, pyarrow, matplotlib
print(json.dumps({
 "numpy":numpy.__version__,"pandas":pandas.__version__,
 "pyarrow":pyarrow.__version__,"matplotlib":matplotlib.__version__
}, sort_keys=True))
"""
probe_proc = _subprocess.run([_sys.executable, "-c", _probe], capture_output=True, text=True)
if probe_proc.returncode != 0:
    raise RuntimeError(
        "WHAT FAILED: post-setup import/API validation failed. "
        "WHY: the runtime is still incompatible or a required API is unavailable. "
        "SAFE TO CONTINUE: NO. Restart with a fresh Kaggle Python runtime rather than upgrading the scientific stack.\n"
        f"probe stdout:\n{probe_proc.stdout[-4000:]}\n"
        f"probe stderr:\n{probe_proc.stderr[-4000:]}"
    )

ENVIRONMENT_PREFLIGHT = {
    "schema_version": 1,
    "environment_id": STAGE_ENVIRONMENT_ID,
    "stage_id": "06",
    "policy": "preserve Kaggle compiled scientific stack; install only missing application-level packages",
    "accelerator_note": 'CPU sufficient',
    "initial_versions": initial_versions,
    "installed_now": installed_now,
    "validated_versions": {name:_dist_version(name) for name in list(CORE_DISTRIBUTIONS)+[x[0] for x in APPLICATION_REQUIREMENTS]},
    "probe_stdout": probe_proc.stdout.strip(),
}
print("Environment validation: PASS")
print(probe_proc.stdout.strip())


Environment validation: PASS
{"matplotlib": "3.10.0", "numpy": "2.0.2", "pandas": "2.3.3", "pyarrow": "24.0.0"}


In [2]:
from pathlib import Path
from datetime import datetime,timezone
from collections import Counter
import hashlib, importlib.metadata as im, json, os, platform, re, subprocess, sys, time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt

STAGE_ID="06";STAGE_NAME="06_fixed_budget_reliability_evaluation";PROTOCOL_VERSION="SWARABYANJAN-CROSSPLATFORM-v1.0.0"
SEEDS=[42,137,2718,31415,65537];MODEL_KEYS=["ebm","tfidf_lr","banglabert","xlmr"];BUDGETS=[0.05,0.10,0.20];PRIMARY_BUDGET=0.10;BOOTSTRAP_REPS=5000;BOOTSTRAP_SEED=20260906
WORK_ROOT=Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd();SEARCH_ROOTS=[Path("/kaggle/input"),WORK_ROOT,Path.cwd()] if Path("/kaggle/input").exists() else [WORK_ROOT,Path.cwd()]
OUT_ROOT=WORK_ROOT/"03_results"/STAGE_NAME
if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()):raise RuntimeError(f"Refusing overwrite: {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True,exist_ok=True)
def utc_now():return datetime.now(timezone.utc).isoformat()
def sha256_file(p,chunk=8*1024*1024):
 h=hashlib.sha256();f=open(p,"rb")
 while True:
  b=f.read(chunk)
  if not b:f.close();return h.hexdigest()
  h.update(b)
def write_json(p,o):Path(p).write_text(json.dumps(o,ensure_ascii=False,sort_keys=True,indent=2)+"\n",encoding="utf-8",newline="\n")
def write_text(p,t):Path(p).write_text(t,encoding="utf-8",newline="\n")
def set_status(s,m,d=None):write_json(OUT_ROOT/"STAGE_STATUS.json",{"stage_id":STAGE_ID,"status":s,"message":m,"timestamp_utc":utc_now(),"details":d or {}});print(f"STAGE 06 STATUS: {s} — {m}")
def hard_stop(m,d=None):
    set_status("HARD_STOP",m,d)
    if d:
        detail=json.dumps(d,ensure_ascii=False,sort_keys=True,default=str)
        raise RuntimeError(f"{m} | details={detail[:12000]}")
    raise RuntimeError(m)
def pkg(n):
 try:return im.version(n)
 except:return None
def git_state():
 try:
  head=subprocess.check_output(["git","rev-parse","HEAD"],stderr=subprocess.DEVNULL,text=True).strip();status=subprocess.check_output(["git","status","--porcelain"],stderr=subprocess.DEVNULL,text=True)
  return {"available":True,"head":head,"dirty":bool(status.strip()),"status_porcelain":status[:20000]}
 except Exception:return {"available":False,"head":None,"dirty":None,"status_porcelain":None}
write_json(OUT_ROOT/"ENVIRONMENT_PREFLIGHT.json",ENVIRONMENT_PREFLIGHT)
set_status("RUNNING","Stage-06 initialization complete")


STAGE 06 STATUS: RUNNING — Stage-06 initialization complete


## Dataset/Input Discovery
Stage 06 discovers the successful Stage-05 output directory directly.

## Upstream Prediction Contract
The three prediction Parquet hashes must agree simultaneously across:
- the original Stage-05 artifact manifest;
- the original Stage-05 checksum contract where declared; and
- the physical files actually loaded by Stage 06.

Across model/seed replicas, Stage 06 also verifies sorted `row_id`, `label`, `cluster_id`, and target-platform identity before H4 analysis.

H3 result artifacts are deliberately not read by Stage 06 because H4 is computed only from the locked Stage-05 prediction Parquets. Therefore a stale H3 summary cannot contaminate Stage-06 inference.


In [3]:
# ---------- Direct canonical Stage-05 prediction contract ----------
EXPECTED_STAGE05_ID = "05"
EXPECTED_STAGE05_CANONICAL_VERSION = "SWARABYANJAN-STAGE05-CANONICAL-vFINAL-1.0.1"
EXPECTED_STAGE05_PROTOCOL_VERSION = "SWARABYANJAN-CROSSPLATFORM-v1.0.0"

def canonical_stage05_identity_matches(man):
    return (
        man.get("stage_id") == EXPECTED_STAGE05_ID
        and man.get("canonical_stage05") is True
        and man.get("canonical_version") == EXPECTED_STAGE05_CANONICAL_VERSION
        and man.get("protocol_version") == EXPECTED_STAGE05_PROTOCOL_VERSION
    )

def verify_canonical_stage05_identity(man, label):
    expected = {
        "stage_id": EXPECTED_STAGE05_ID,
        "canonical_stage05": True,
        "canonical_version": EXPECTED_STAGE05_CANONICAL_VERSION,
        "protocol_version": EXPECTED_STAGE05_PROTOCOL_VERSION,
    }
    observed = {k: man.get(k) for k in expected}
    mismatches = {}
    for k, exp in expected.items():
        obs = observed[k]
        ok = (obs is True) if k == "canonical_stage05" else (obs == exp)
        if not ok:
            mismatches[k] = {"expected": exp, "observed": obs}
    if mismatches:
        hard_stop(
            "Stage-05 canonical identity/version mismatch",
            {"manifest": label, "mismatches": mismatches, "safe_to_continue": False},
        )
    return expected

def cjb(o):
    return json.dumps(o,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode("utf-8")

def parse_sum(path):
    out={}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        parts=line.split(None,1)
        if len(parts)!=2: hard_stop("Malformed checksum line",{"source":str(path),"line":line})
        h,r=parts
        r=r.lstrip("*").strip().replace("\\","/")
        if not re.fullmatch(r"[0-9a-f]{64}",h.lower()): hard_stop("Malformed checksum hash",{"source":str(path),"line":line})
        pp=Path(r)
        if pp.is_absolute() or ".." in pp.parts: hard_stop("Unsafe checksum path",{"source":str(path),"path":r})
        if r in out: hard_stop("Duplicate checksum path",{"source":str(path),"path":r})
        out[r]=h.lower()
    return out

def verify_manifest_payload(man,label):
    recorded=man.get("manifest_payload_sha256")
    if not recorded:
        hard_stop("Stage-05 manifest payload seal missing",{"manifest":label})
    tmp=dict(man);tmp.pop("manifest_payload_sha256",None)
    observed=hashlib.sha256(cjb(tmp)).hexdigest()
    if observed!=recorded:
        hard_stop("Stage-05 manifest payload seal mismatch",{"manifest":label,"recorded":recorded,"observed":observed})

def candidate_identity(root):
    root=Path(root)
    required=[
        "STAGE_STATUS.json","STAGE05_ARTIFACT_MANIFEST.json","STAGE05_SHA256SUMS.txt",
        "LOCKED_SOURCE_TEST_PREDICTIONS.parquet","BANGLACLICK_PLATFORM_PREDICTIONS.parquet","BAITBUSTER_HUMAN_PREDICTIONS.parquet",
    ]
    if not all((root/x).is_file() for x in required): return None
    try:
        man=json.loads((root/"STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
        if not canonical_stage05_identity_matches(man):
            return None
        man_sha=sha256_file(root/"STAGE05_ARTIFACT_MANIFEST.json")
        sums_sha=sha256_file(root/"STAGE05_SHA256SUMS.txt")
        pred_sha={x:sha256_file(root/x) for x in required[3:]}
    except Exception:
        return None
    return {
        "root":root,
        "manifest_sha256":man_sha,
        "checksum_sha256":sums_sha,
        "prediction_sha256":pred_sha,
        "stage05_identity":{
            "stage_id":man.get("stage_id"),
            "canonical_stage05":man.get("canonical_stage05"),
            "canonical_version":man.get("canonical_version"),
            "protocol_version":man.get("protocol_version"),
        },
    }

def discover_stage05_root():
    candidates=[];seen=set()
    for search_root in SEARCH_ROOTS:
        if not search_root.exists(): continue
        for p in search_root.rglob("STAGE05_ARTIFACT_MANIFEST.json"):
            d=p.parent.resolve()
            if d in seen: continue
            seen.add(d)
            rec=candidate_identity(d)
            if rec is not None: candidates.append(rec)
    if not candidates:
        hard_stop("Canonical final Stage-05 output directory not found",{
            "required":["STAGE_STATUS.json","STAGE05_ARTIFACT_MANIFEST.json","STAGE05_SHA256SUMS.txt",
                        "LOCKED_SOURCE_TEST_PREDICTIONS.parquet","BANGLACLICK_PLATFORM_PREDICTIONS.parquet","BAITBUSTER_HUMAN_PREDICTIONS.parquet"],
            "safe_to_continue":False,
        })
    # Duplicate mounts of the same canonical Stage-05 scientific identity are harmless; distinct canonical identities are not.
    by_identity={}
    for rec in candidates:
        ident=(rec["manifest_sha256"],rec["checksum_sha256"],tuple(sorted(rec["prediction_sha256"].items())))
        by_identity.setdefault(ident,[]).append(rec)
    if len(by_identity)!=1:
        hard_stop("Multiple distinct canonical Stage-05 output identities found",{
            "candidates":[{"root":str(r["root"]),"manifest_sha256":r["manifest_sha256"]} for r in candidates],
            "safe_to_continue":False,
        })
    group=next(iter(by_identity.values()))
    chosen=sorted(group,key=lambda r:str(r["root"]))[0]
    if len(group)>1:
        print(f"Canonical Stage-05 output: identical mounts collapsed ({len(group)}); using {chosen['root']}")
    return chosen

S5_REC=discover_stage05_root();S5=S5_REC["root"]
MAN=json.loads((S5/"STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
verify_manifest_payload(MAN,str(S5/"STAGE05_ARTIFACT_MANIFEST.json"))
STAGE05_CANONICAL_IDENTITY = verify_canonical_stage05_identity(
    MAN, str(S5/"STAGE05_ARTIFACT_MANIFEST.json")
)
S5_STATUS=json.loads((S5/"STAGE_STATUS.json").read_text(encoding="utf-8"))
if S5_STATUS.get("status")!="PASS":
    hard_stop("Underlying Stage-05 prediction run is not PASS",{"status":S5_STATUS})

sums=parse_sum(S5/"STAGE05_SHA256SUMS.txt")
fail=[]
for rel,h in sorted(sums.items()):
    p=S5/rel
    if not p.is_file(): fail.append((rel,"missing"))
    elif sha256_file(p)!=h: fail.append((rel,"hash"))
if fail:
    hard_stop("Stage-05 checksum verification failed",{"failures":fail[:20]})

PRED_FILES=["LOCKED_SOURCE_TEST_PREDICTIONS.parquet","BANGLACLICK_PLATFORM_PREDICTIONS.parquet","BAITBUSTER_HUMAN_PREDICTIONS.parquet"]
physical_prediction_hashes={}
for x in PRED_FILES:
    p=S5/x
    actual=sha256_file(p);physical_prediction_hashes[x]=actual
    if MAN.get("artifacts",{}).get(x)!=actual:
        hard_stop("Prediction hash disagrees with Stage-05 manifest",{"artifact":x})
    if x in sums and sums[x]!=actual:
        hard_stop("Prediction hash disagrees with Stage-05 checksum contract",{"artifact":x})

SRC=pd.read_parquet(S5/PRED_FILES[0]);BC=pd.read_parquet(S5/PRED_FILES[1]);BB=pd.read_parquet(S5/PRED_FILES[2])

def verify_replica_invariants(df,name,target):
    needed={"row_id","cluster_id","label","model","seed","p_uncalibrated","p_calibrated"}
    if target: needed.add("platform")
    if not needed.issubset(df.columns):
        hard_stop("Prediction schema mismatch",{"dataset":name,"missing":sorted(needed-set(df.columns))})
    if df.empty: hard_stop("Prediction dataset is empty",{"dataset":name})
    if set(df["label"].dropna().astype(int).unique())!={0,1}:
        hard_stop("Prediction labels are not binary with both classes",{"dataset":name})
    if not np.isfinite(df["p_calibrated"].to_numpy(float)).all() or ((df["p_calibrated"]<0)|(df["p_calibrated"]>1)).any():
        hard_stop("Calibrated probabilities are invalid",{"dataset":name})
    panel=set(zip(df["model"].astype(str),df["seed"].astype(int)))
    expected_panel={(m,s) for m in MODEL_KEYS for s in SEEDS}
    if panel!=expected_panel:
        hard_stop("Prediction model/seed panel mismatch",{"dataset":name,"missing":sorted(map(str,expected_panel-panel)),"extra":sorted(map(str,panel-expected_panel))})
    canonical=None
    static_cols=["row_id","label","cluster_id"]+(["platform"] if target else [])
    for (m,s),g in df.groupby(["model","seed"],sort=True):
        x=g.sort_values("row_id",kind="stable").reset_index(drop=True)
        if x["row_id"].duplicated().any():
            hard_stop("Duplicate row IDs within prediction group",{"dataset":name,"model":m,"seed":int(s)})
        cur=x[static_cols].copy()
        cur["row_id"]=cur["row_id"].astype(str);cur["cluster_id"]=cur["cluster_id"].astype(str);cur["label"]=cur["label"].astype(int)
        if target: cur["platform"]=cur["platform"].astype(str)
        if canonical is None: canonical=cur
        elif not cur.equals(canonical):
            hard_stop("Prediction row/label/cluster/platform invariants differ across model/seeds",{"dataset":name,"model":m,"seed":int(s)})
    return canonical

verify_replica_invariants(SRC,"source_test",False)
verify_replica_invariants(BC,"banglaclick",True)
verify_replica_invariants(BB,"baitbuster_human",True)

observed=set(BC["platform"].astype(str).unique());expected={"news","facebook","youtube","twitter"}
if observed!=expected:
    hard_stop("BanglaClick platform contract mismatch",{"observed":sorted(observed),"expected":sorted(expected)})

# H3 result artifacts are intentionally not read. H4 depends only on locked predictions.
PROVENANCE_GATE={
    "schema_version":2,
    "status":"PASS",
    "mode":"DIRECT_CANONICAL_STAGE05_OUTPUT",
    "stage05_root":str(S5),
    "stage05_manifest_sha256":sha256_file(S5/"STAGE05_ARTIFACT_MANIFEST.json"),
    "stage05_checksum_sha256":sha256_file(S5/"STAGE05_SHA256SUMS.txt"),
    "stage05_status_sha256":sha256_file(S5/"STAGE_STATUS.json"),
    "stage05_status":"PASS",
    "stage05_identity":STAGE05_CANONICAL_IDENTITY,
    "prediction_sha256":physical_prediction_hashes,
    "h3_statistics_consumed":False,
    "h3_summary_consumed":False,
    "reason_h3_not_consumed":"Stage 06 H4 uses immutable prediction artifacts only; H3 statistics do not enter H4 computation.",
}
write_json(OUT_ROOT/"UPSTREAM_PROVENANCE_GATE.json",PROVENANCE_GATE)
print("Direct canonical Stage-05 provenance gate: PASS")
print("Stage-05 prediction-only contract and replica invariants: PASS")

# Verify scalar temperature scaling preserves positive-risk order before H4 analysis.
ranking_checks=[]
for dname,df in [("source_test",SRC),("banglaclick",BC),("baitbuster_human",BB)]:
    for (m,s),g in df.groupby(["model","seed"],sort=True):
        a=g.sort_values(["p_uncalibrated","row_id"],ascending=[False,True],kind="stable")["row_id"].astype(str).tolist()
        b=g.sort_values(["p_calibrated","row_id"],ascending=[False,True],kind="stable")["row_id"].astype(str).tolist()
        ok=(a==b);ranking_checks.append({"dataset":dname,"model":m,"seed":int(s),"same_order":ok})
        if not ok:
            hard_stop("Positive-risk ranking changed after scalar temperature scaling",{"dataset":dname,"model":m,"seed":int(s),"safe_to_continue":False})
print("Temperature-scaling positive-risk ranking invariant: PASS")


Direct canonical Stage-05 provenance gate: PASS
Stage-05 prediction-only contract and replica invariants: PASS
Temperature-scaling positive-risk ranking invariant: PASS


## Method
Positive-risk triage ranks frozen calibrated positive probabilities; uncertainty review ranks `1 - max(p, 1-p)`. These are distinct operational tasks.

## Evaluation
Fixed budgets are 5%, 10%, and 20%, with 10% primary. Positive-risk precision/recall/lift, selective-error capture, risk–coverage, and AURC are reported.

### Fixed-Budget and Selective-Risk Definitions
Positive triage ranks calibrated positive-class risk. Error review ranks predictive uncertainty. The tasks are evaluated separately at all frozen budgets.


In [4]:
# ---------- Fixed-budget and selective-risk definitions; precomputed once per model/seed/domain ----------
def uncertainty_score(p):
    return 1-np.maximum(p,1-p)

class ClusterBootstrapPlan:
    def __init__(self,cluster_ids,name):
        a=np.asarray(cluster_ids).astype(str)
        if len(a)==0: raise ValueError(f"{name}: empty cluster vector")
        u=np.array(sorted(set(a)))
        code={x:i for i,x in enumerate(u)}
        self.row_cluster=np.asarray([code[x] for x in a],dtype=np.int32)
        self.cluster_labels=u
        self.n_clusters=int(len(u))
        self.n_rows=int(len(a))
        self.name=str(name)
    def draw_row_counts(self,rng):
        sampled=rng.choice(self.n_clusters,size=self.n_clusters,replace=True)
        cluster_mult=np.bincount(sampled,minlength=self.n_clusters)
        return cluster_mult[self.row_cluster].astype(np.int64,copy=False)

_HARMONIC=np.array([0.0],dtype=float)
def _ensure_harmonic(n):
    global _HARMONIC
    n=int(n)
    if n < len(_HARMONIC): return
    start=len(_HARMONIC)
    x=np.arange(start,n+1,dtype=float)
    extra=_HARMONIC[-1]+np.cumsum(1.0/x)
    _HARMONIC=np.concatenate([_HARMONIC,extra])

def _weighted_prefix_sum(cum_w,cum_v,values,take):
    take=int(take)
    if take<=0: return 0.0
    j=int(np.searchsorted(cum_w,take,side="left"))
    prev_w=float(cum_w[j-1]) if j else 0.0
    prev_v=float(cum_v[j-1]) if j else 0.0
    return prev_v+(take-prev_w)*float(values[j])

class OperationalPlan:
    def __init__(self,g,name):
        frame=g.sort_values("row_id",kind="stable").reset_index(drop=True).copy()
        if frame.empty: raise ValueError(f"{name}: empty prediction group")
        self.name=str(name);self.frame=frame;self.n=len(frame)
        self.row_id=frame["row_id"].astype(str).to_numpy()
        self.y=frame["label"].astype(int).to_numpy()
        self.p=frame["p_calibrated"].astype(float).to_numpy()
        self.error=((self.p>=0.5).astype(int)!=self.y).astype(np.int8)
        self.uncertainty=uncertainty_score(self.p)
        # These rankings are computed ONCE; no sorting occurs in bootstrap replicates.
        tmp=pd.DataFrame({"i":np.arange(self.n),"row_id":self.row_id,"p":self.p,"u":self.uncertainty})
        self.pos_order=tmp.sort_values(["p","row_id"],ascending=[False,True],kind="stable")["i"].to_numpy(int)
        self.unc_high_order=tmp.sort_values(["u","row_id"],ascending=[False,True],kind="stable")["i"].to_numpy(int)
        self.unc_low_order=tmp.sort_values(["u","row_id"],ascending=[True,True],kind="stable")["i"].to_numpy(int)
        self.cluster=ClusterBootstrapPlan(frame["cluster_id"].astype(str).to_numpy(),name)
        self.ones=np.ones(self.n,dtype=np.int64)

    def metrics(self,row_counts):
        w=np.asarray(row_counts,dtype=np.int64)
        if w.shape!=(self.n,) or (w<0).any():
            raise ValueError(f"{self.name}: invalid row multiplicities")
        n=int(w.sum())
        if n<=0: raise ValueError(f"{self.name}: bootstrap resample contains zero rows")
        total_pos=float(np.dot(self.y,w))
        total_err=float(np.dot(self.error,w))
        prevalence=total_pos/n

        # Positive-risk triage weighted prefixes.
        pw=w[self.pos_order]
        pos_cw=np.cumsum(pw)
        pos_cy=np.cumsum(pw*self.y[self.pos_order])

        # Uncertainty review weighted error prefixes.
        uw=w[self.unc_high_order]
        unc_cw=np.cumsum(uw)
        unc_ce=np.cumsum(uw*self.error[self.unc_high_order])

        budget_out={}
        for k in BUDGETS:
            take=max(1,int(np.ceil(float(k)*n)))
            captured_pos=_weighted_prefix_sum(pos_cw,pos_cy,self.y[self.pos_order],take)
            captured_err=_weighted_prefix_sum(unc_cw,unc_ce,self.error[self.unc_high_order],take)
            precision=float(captured_pos/take)
            recall=float(captured_pos/total_pos) if total_pos>0 else np.nan
            lift=float(precision/prevalence) if total_pos>0 else np.nan
            error_capture=float(captured_err/total_err) if total_err>0 else np.nan
            budget_out[float(k)]={
                "reviewed":int(take),
                "precision":precision,
                "recall":recall,
                "lift":lift,
                "error_capture":error_capture,
                "recall_estimability_status":"ESTIMABLE" if total_pos>0 else "NON_ESTIMABLE_TOTAL_POS_ZERO",
                "lift_estimability_status":"ESTIMABLE" if total_pos>0 else "NON_ESTIMABLE_TOTAL_POS_ZERO",
                "error_capture_estimability_status":"ESTIMABLE" if total_err>0 else "NON_ESTIMABLE_TOTAL_ERR_ZERO",
            }

        # Exact compressed equivalent of the original trapezoidal risk–coverage AURC.
        lw=w[self.unc_low_order]
        le=self.error[self.unc_low_order].astype(np.int64,copy=False)
        cw=np.cumsum(lw)
        before=cw-lw
        prior_err=np.cumsum(lw*le)-lw*le
        _ensure_harmonic(n)
        dH=_HARMONIC[cw]-_HARMONIC[before]
        sum_risk=float(np.where(le==0,prior_err*dH,lw+(prior_err-before)*dH).sum())
        first_idx=int(np.argmax(lw>0))
        first_risk=float(le[first_idx])
        final_risk=float(total_err/n)
        aurc=first_risk if n==1 else float((sum_risk-0.5*first_risk-0.5*final_risk)/n)
        # Any unexpected non-finite AURC is propagated as NaN so the bootstrap
        # layer can count the replicate invalid and record the explicit reason.
        if not np.isfinite(aurc): aurc=np.nan
        state={"n":n,"total_pos":total_pos,"total_err":total_err,"prevalence":prevalence}
        return budget_out,aurc,state

    def point_risk_curve(self):
        order=self.unc_low_order
        err=self.error[order].astype(float)
        cov=np.arange(1,self.n+1,dtype=float)/self.n
        risk=np.cumsum(err)/np.arange(1,self.n+1,dtype=float)
        return pd.DataFrame({"coverage":cov,"risk":risk})

domains={"source_test":SRC,"baitbuster:youtube_human":BB}
for p in ["news","facebook","youtube","twitter"]:
    domains[f"banglaclick:{p}"]=BC[BC["platform"].eq(p)]

PLANS={};POINT_METRICS={}
rows=[];rc_rows=[]
for dname,df in domains.items():
    for (m,s),g in df.groupby(["model","seed"],sort=True):
        key=(dname,str(m),int(s))
        plan=OperationalPlan(g,f"{dname}|{m}|{int(s)}")
        PLANS[key]=plan
        bud,aurc,state=plan.metrics(plan.ones)
        POINT_METRICS[key]=(bud,aurc,state)
        for k in BUDGETS:
            z=bud[float(k)]
            rows.append({
                "domain":dname,"model":str(m),"seed":int(s),"n":plan.n,
                "prevalence":float(state["prevalence"]),"budget":float(k),
                **z,"aurc":aurc,"primary_budget":float(k)==PRIMARY_BUDGET,
            })
        rc=plan.point_risk_curve()
        rr=rc.iloc[np.unique(np.linspace(0,len(rc)-1,min(201,len(rc))).astype(int))].copy()
        rr["domain"]=dname;rr["model"]=str(m);rr["seed"]=int(s);rc_rows.append(rr)

BUDGET=pd.DataFrame(rows).sort_values(["domain","model","seed","budget"],kind="stable")
RC=pd.concat(rc_rows,ignore_index=True)
BUDGET.to_csv(OUT_ROOT/"FIXED_BUDGET_TABLES.csv",index=False)
RC.to_csv(OUT_ROOT/"RISK_COVERAGE_DATA.csv",index=False)
print(f"Operational plans precomputed: {len(PLANS)} model/seed/domain groups")


Operational plans precomputed: 120 model/seed/domain groups


## Statistical Analysis
H4 source-to-target differences use the locked 5,000 cluster-bootstrap replicates with model/seed pairing preserved.


In [5]:
# ---------- Cluster-bootstrap H4 source-to-target operational deltas ----------
def finite_or_none(x):
    try:
        x=float(x)
        return x if np.isfinite(x) else None
    except (TypeError,ValueError):
        return None

def delta_if_finite(target,source):
    t=finite_or_none(target);s=finite_or_none(source)
    return (t-s) if t is not None and s is not None else np.nan

def complete_ci(a):
    a=np.asarray(a,float)
    if len(a)!=BOOTSTRAP_REPS:
        raise ValueError(f"CI requires exactly {BOOTSTRAP_REPS} locked bootstrap attempts")
    if not np.isfinite(a).all():
        return None
    return [float(np.percentile(a,2.5)),float(np.percentile(a,97.5))]

def complete_p2(a):
    a=np.asarray(a,float)
    if len(a)!=BOOTSTRAP_REPS:
        raise ValueError(f"p-value requires exactly {BOOTSTRAP_REPS} locked bootstrap attempts")
    if not np.isfinite(a).all():
        return None
    return float(min(1.0,2*min(np.mean(a<=0),np.mean(a>=0))))

def finalize_metric(values,reason_counts):
    a=np.asarray(values,float)
    if len(a)!=BOOTSTRAP_REPS:
        raise ValueError("Bootstrap result length mismatch")
    valid=int(np.isfinite(a).sum());invalid=int(BOOTSTRAP_REPS-valid)
    if int(sum(reason_counts.values()))!=invalid:
        raise ValueError("Invalid-reason accounting does not equal invalid replicate count")
    estimable=(invalid==0)
    return {
        "bootstrap_attempts":BOOTSTRAP_REPS,
        "valid_replicates":valid,
        "invalid_replicates":invalid,
        "invalid_rate":float(invalid/BOOTSTRAP_REPS),
        "invalid_reason_counts":dict(sorted(reason_counts.items())),
        "estimability_status":"ESTIMABLE" if estimable else "NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP",
        "ci95":complete_ci(a) if estimable else None,
        "p_two_sided":complete_p2(a) if estimable else None,
    }

def point_metric_delta(target_value,source_value):
    t=finite_or_none(target_value);s=finite_or_none(source_value)
    return (t-s) if t is not None and s is not None else None

rng=np.random.default_rng(BOOTSTRAP_SEED)
inf=[];setting_runtime=[]
target_domains=[d for d in domains if d!="source_test"]
total_settings=len(target_domains)*len(MODEL_KEYS)*len(SEEDS)
phase_start=time.monotonic();setting_no=0
print(f"[H4 bootstrap] START — settings={total_settings}, replicates_per_setting={BOOTSTRAP_REPS}",flush=True)

for dname in target_domains:
    for m in MODEL_KEYS:
        for seed in SEEDS:
            setting_no+=1
            setting_start=time.monotonic()
            skey=("source_test",m,int(seed));tkey=(dname,m,int(seed))
            sp=PLANS[skey];tp=PLANS[tkey]
            point_sb,point_sa,point_ss=POINT_METRICS[skey]
            point_tb,point_ta,point_ts=POINT_METRICS[tkey]
            store={
                float(k):{
                    "lift":[],
                    "error_capture":[],
                    "lift_reasons":Counter(),
                    "error_capture_reasons":Counter(),
                } for k in BUDGETS
            }
            aurc_values=[];aurc_reasons=Counter()
            detail=f"setting={setting_no}/{total_settings} target={dname} model={m} seed={seed}"
            print(f"[H4 bootstrap] SETTING START — {detail}",flush=True)

            for rep in range(1,BOOTSTRAP_REPS+1):
                sw=sp.cluster.draw_row_counts(rng)
                tw=tp.cluster.draw_row_counts(rng)
                sb,sa,ss=sp.metrics(sw)
                tb,ta,ts=tp.metrics(tw)

                for k in BUDGETS:
                    kk=float(k)
                    # Lift is undefined whenever source or target bootstrap prevalence is zero.
                    lift_reasons=[]
                    if ss["total_pos"]<=0: lift_reasons.append("source_total_pos_zero")
                    if ts["total_pos"]<=0: lift_reasons.append("target_total_pos_zero")
                    if lift_reasons:
                        store[kk]["lift"].append(np.nan)
                        store[kk]["lift_reasons"]["+".join(sorted(lift_reasons))]+=1
                    else:
                        dv=delta_if_finite(tb[kk]["lift"],sb[kk]["lift"])
                        if np.isfinite(dv):
                            store[kk]["lift"].append(float(dv))
                        else:
                            store[kk]["lift"].append(np.nan)
                            store[kk]["lift_reasons"]["other_nonfinite_lift_endpoint"]+=1

                    # Error capture is undefined whenever source or target has zero errors.
                    error_reasons=[]
                    if ss["total_err"]<=0: error_reasons.append("source_total_err_zero")
                    if ts["total_err"]<=0: error_reasons.append("target_total_err_zero")
                    if error_reasons:
                        store[kk]["error_capture"].append(np.nan)
                        store[kk]["error_capture_reasons"]["+".join(sorted(error_reasons))]+=1
                    else:
                        dv=delta_if_finite(tb[kk]["error_capture"],sb[kk]["error_capture"])
                        if np.isfinite(dv):
                            store[kk]["error_capture"].append(float(dv))
                        else:
                            store[kk]["error_capture"].append(np.nan)
                            store[kk]["error_capture_reasons"]["other_nonfinite_error_capture_endpoint"]+=1

                ad=delta_if_finite(ta,sa)
                if np.isfinite(ad):
                    aurc_values.append(float(ad))
                else:
                    aurc_values.append(np.nan)
                    aurc_reasons["other_nonfinite_aurc_endpoint"]+=1

                if rep%1000==0 or rep==BOOTSTRAP_REPS:
                    elapsed=time.monotonic()-setting_start
                    rate=rep/elapsed if elapsed>0 else None
                    eta=(BOOTSTRAP_REPS-rep)/rate if rate and rate>0 else None
                    print(
                        f"[H4 bootstrap] {detail} {rep}/{BOOTSTRAP_REPS} "
                        f"elapsed={elapsed:.1f}s eta={0.0 if eta is None else eta:.1f}s",
                        flush=True,
                    )

            aurc_final=finalize_metric(aurc_values,aurc_reasons)
            for k in BUDGETS:
                kk=float(k)
                lift_final=finalize_metric(store[kk]["lift"],store[kk]["lift_reasons"])
                error_final=finalize_metric(store[kk]["error_capture"],store[kk]["error_capture_reasons"])

                source_lift=finite_or_none(point_sb[kk]["lift"])
                target_lift=finite_or_none(point_tb[kk]["lift"])
                source_error=finite_or_none(point_sb[kk]["error_capture"])
                target_error=finite_or_none(point_tb[kk]["error_capture"])

                inf.append({
                    "domain":dname,"model":m,"seed":int(seed),"budget":kk,
                    "primary_budget":kk==PRIMARY_BUDGET,
                    "source_lift":source_lift,
                    "target_lift":target_lift,
                    "target_minus_source_lift":point_metric_delta(target_lift,source_lift),
                    "lift_ci95":lift_final["ci95"],
                    "lift_p":lift_final["p_two_sided"],
                    "lift_bootstrap_attempts":lift_final["bootstrap_attempts"],
                    "lift_valid_replicates":lift_final["valid_replicates"],
                    "lift_invalid_replicates":lift_final["invalid_replicates"],
                    "lift_invalid_rate":lift_final["invalid_rate"],
                    "lift_invalid_reason_counts":lift_final["invalid_reason_counts"],
                    "lift_estimability_status":lift_final["estimability_status"],

                    "source_error_capture":source_error,
                    "target_error_capture":target_error,
                    "target_minus_source_error_capture":point_metric_delta(target_error,source_error),
                    "error_capture_ci95":error_final["ci95"],
                    "error_capture_p":error_final["p_two_sided"],
                    "error_capture_bootstrap_attempts":error_final["bootstrap_attempts"],
                    "error_capture_valid_replicates":error_final["valid_replicates"],
                    "error_capture_invalid_replicates":error_final["invalid_replicates"],
                    "error_capture_invalid_rate":error_final["invalid_rate"],
                    "error_capture_invalid_reason_counts":error_final["invalid_reason_counts"],
                    "error_capture_estimability_status":error_final["estimability_status"],

                    "source_aurc":finite_or_none(point_sa),
                    "target_aurc":finite_or_none(point_ta),
                    "target_minus_source_aurc":point_metric_delta(point_ta,point_sa),
                    "aurc_ci95":aurc_final["ci95"],
                    "aurc_p":aurc_final["p_two_sided"],
                    "aurc_bootstrap_attempts":aurc_final["bootstrap_attempts"],
                    "aurc_valid_replicates":aurc_final["valid_replicates"],
                    "aurc_invalid_replicates":aurc_final["invalid_replicates"],
                    "aurc_invalid_rate":aurc_final["invalid_rate"],
                    "aurc_invalid_reason_counts":aurc_final["invalid_reason_counts"],
                    "aurc_estimability_status":aurc_final["estimability_status"],
                    "bootstrap_attempts":BOOTSTRAP_REPS,
                })

            setting_elapsed=time.monotonic()-setting_start
            setting_runtime.append({
                "domain":dname,"model":m,"seed":int(seed),
                "bootstrap_replicates":BOOTSTRAP_REPS,
                "elapsed_seconds":float(setting_elapsed),
            })
            print(f"[H4 bootstrap] SETTING COMPLETE — {detail} elapsed={setting_elapsed:.1f}s",flush=True)

bootstrap_total_seconds=float(time.monotonic()-phase_start)
BOOTSTRAP_RUNTIME={
    "schema_version":1,
    "bootstrap_seed":BOOTSTRAP_SEED,
    "bootstrap_replicates_per_setting":BOOTSTRAP_REPS,
    "settings":len(setting_runtime),
    "total_elapsed_seconds":bootstrap_total_seconds,
    "setting_runtime":setting_runtime,
}
write_json(OUT_ROOT/"BOOTSTRAP_RUNTIME.json",BOOTSTRAP_RUNTIME)

def json_safe_records(df):
    out=[]
    for rec in df.to_dict(orient="records"):
        clean={}
        for k,v in rec.items():
            if isinstance(v,(np.integer,)): v=int(v)
            elif isinstance(v,(np.floating,)): v=float(v)
            if isinstance(v,float) and not np.isfinite(v): v=None
            clean[k]=v
        out.append(clean)
    return out

ESTIMABILITY_POLICY={
    "bootstrap_attempts_required":BOOTSTRAP_REPS,
    "conditional_surviving_draw_inference":False,
    "rejection_resampling":False,
    "lift_invalid_when":"source_total_pos==0 or target_total_pos==0, or any other non-finite lift endpoint",
    "error_capture_invalid_when":"source_total_err==0 or target_total_err==0, or any other non-finite error-capture endpoint",
    "aurc_invalid_when":"any non-finite AURC endpoint",
    "confirmatory_rule":"CI and two-sided bootstrap p-value are reported only when all 5,000 locked replicates are valid for that metric.",
}
write_json(
    OUT_ROOT/"H4_RESULTS.json",
    {
        "schema_version":2,
        "positive_triage":"rank p(y=1); precision/recall/lift",
        "selective_review":"rank 1-max(p,1-p); error capture + risk coverage/AURC",
        "budgets":BUDGETS,
        "primary_budget":PRIMARY_BUDGET,
        "temperature_ranking_claim":"Scalar temperature is strictly monotone for T>0; it is not credited with changing pure positive-risk top-k ordering.",
        "bootstrap_seed":BOOTSTRAP_SEED,
        "bootstrap_replicates":BOOTSTRAP_REPS,
        "estimability_policy":ESTIMABILITY_POLICY,
        "fixed_budget_point_estimates":json_safe_records(BUDGET),
        "bootstrap_rows":inf,
        "runtime":{"bootstrap_total_seconds":bootstrap_total_seconds},
    },
)
print(f"[H4 bootstrap] COMPLETE — total elapsed={bootstrap_total_seconds:.1f}s",flush=True)


[H4 bootstrap] START — settings=100, replicates_per_setting=5000
[H4 bootstrap] SETTING START — setting=1/100 target=baitbuster:youtube_human model=ebm seed=42
[H4 bootstrap] setting=1/100 target=baitbuster:youtube_human model=ebm seed=42 1000/5000 elapsed=1.1s eta=4.5s
[H4 bootstrap] setting=1/100 target=baitbuster:youtube_human model=ebm seed=42 2000/5000 elapsed=2.2s eta=3.4s
[H4 bootstrap] setting=1/100 target=baitbuster:youtube_human model=ebm seed=42 3000/5000 elapsed=3.3s eta=2.2s
[H4 bootstrap] setting=1/100 target=baitbuster:youtube_human model=ebm seed=42 4000/5000 elapsed=4.5s eta=1.1s
[H4 bootstrap] setting=1/100 target=baitbuster:youtube_human model=ebm seed=42 5000/5000 elapsed=5.6s eta=0.0s
[H4 bootstrap] SETTING COMPLETE — setting=1/100 target=baitbuster:youtube_human model=ebm seed=42 elapsed=5.6s
[H4 bootstrap] SETTING START — setting=2/100 target=baitbuster:youtube_human model=ebm seed=137
[H4 bootstrap] setting=2/100 target=baitbuster:youtube_human model=ebm seed=13

## Output Artifacts
Budget tables, risk–coverage data, H4 bootstrap results, figures, runtime metadata, manifests, and checksums are persisted.

## Stage Decision
Risk-coverage curves, fixed-budget summaries, statistical outputs, runtime metadata, manifests, and hashes are persisted before `PASS`.


In [6]:
# ---------- Final tables/figures + statistical summary ----------
figdir=OUT_ROOT/"figures";tabdir=OUT_ROOT/"tables";figdir.mkdir(exist_ok=True);tabdir.mkdir(exist_ok=True)
primary=BUDGET[BUDGET["primary_budget"]].copy()
primary.to_csv(tabdir/"H4_PRIMARY_10_PERCENT_BY_SEED.csv",index=False)
summary=(
    primary.groupby(["domain","model"],sort=True)[["precision","recall","lift","error_capture","aurc"]]
    .agg(["mean","std","count"])
    .reset_index()
)
summary.to_csv(tabdir/"H4_PRIMARY_10_PERCENT_SEED_SUMMARY.csv",index=False)

for m in MODEL_KEYS:
    d=RC[(RC["model"]==m)&(RC["seed"]==SEEDS[0])]
    fig,ax=plt.subplots(figsize=(7,5))
    for name,g in d.groupby("domain",sort=True):
        ax.plot(g["coverage"],g["risk"],label=name)
    ax.set_xlabel("Coverage");ax.set_ylabel("Selective risk")
    ax.set_title(f"Risk–coverage: {m} (seed {SEEDS[0]})")
    ax.legend(fontsize=7);fig.tight_layout()
    fig.savefig(figdir/f"risk_coverage_{m}.png",dpi=180);plt.close(fig)

write_json(
    OUT_ROOT/"FINAL_STATISTICAL_SUMMARY.json",
    {
        "schema_version":2,
        "bootstrap_replicates":BOOTSTRAP_REPS,
        "bootstrap_seed":BOOTSTRAP_SEED,
        "ci":"95% percentile; reported only when all 5,000 locked replicates are valid for the metric",
        "primary_budget":PRIMARY_BUDGET,
        "budgets":BUDGETS,
        "h4_inference":inf,
        "estimability_policy":ESTIMABILITY_POLICY,
        "positive_triage_and_error_review_conflated":False,
        "upstream_provenance_gate_sha256":sha256_file(OUT_ROOT/"UPSTREAM_PROVENANCE_GATE.json"),
        "bootstrap_runtime_sha256":sha256_file(OUT_ROOT/"BOOTSTRAP_RUNTIME.json"),
    },
)


In [7]:
# ---------- Reproducibility manifest and PASS ----------
required=[
    "ENVIRONMENT_PREFLIGHT.json",
    "UPSTREAM_PROVENANCE_GATE.json",
    "H4_RESULTS.json",
    "FIXED_BUDGET_TABLES.csv",
    "RISK_COVERAGE_DATA.csv",
    "BOOTSTRAP_RUNTIME.json",
    "FINAL_STATISTICAL_SUMMARY.json",
]
missing=[x for x in required if not (OUT_ROOT/x).is_file()]
if missing: hard_stop("Stage-06 required artifact missing",{"missing":missing})

RUN={
    "schema_version":2,
    "stage_id":STAGE_ID,
    "execution_utc":utc_now(),
    "python":sys.version,
    "platform":platform.platform(),
    "runtime":{
        "cpu_count":os.cpu_count(),
        "accelerator_required":False,
        "bootstrap_total_seconds":bootstrap_total_seconds,
    },
    "packages":{n:pkg(n) for n in ["pandas","numpy","pyarrow","scikit-learn","matplotlib"]},
    "git":git_state(),
    "input":"Immutable predictions from the directly mounted canonical final Stage-05 output",
    "stage05_manifest_sha256":sha256_file(S5/"STAGE05_ARTIFACT_MANIFEST.json"),
    "stage05_checksum_sha256":sha256_file(S5/"STAGE05_SHA256SUMS.txt"),
    "stage05_status_sha256":sha256_file(S5/"STAGE_STATUS.json"),
    "stage05_identity":STAGE05_CANONICAL_IDENTITY,
    "h3_statistics_consumed":False,
    "prediction_sha256":physical_prediction_hashes,
    "budgets":BUDGETS,
    "primary_budget":PRIMARY_BUDGET,
    "bootstrap_reps":BOOTSTRAP_REPS,
    "bootstrap_seed":BOOTSTRAP_SEED,
    "temperature_monotonic_ranking_validated":all(x["same_order"] for x in ranking_checks),
    "estimability_policy":ESTIMABILITY_POLICY,
}
write_json(OUT_ROOT/"RUN_METADATA.json",RUN)

write_text(
    OUT_ROOT/"STAGE06_SUMMARY.md",
    "# Stage 06 execution summary\n\n"
    f"- Budgets: {BUDGETS}; primary = {PRIMARY_BUDGET:.0%}\n"
    f"- Bootstrap replicates per source-target/model/seed setting: {BOOTSTRAP_REPS}\n"
    f"- Bootstrap seed: {BOOTSTRAP_SEED}\n"
    "- Positive triage and selective/error review were evaluated as distinct operational tasks.\n"
    "- Cluster bootstrap uses precomputed cluster membership and exact row-multiplicity weights; no resampled DataFrames are constructed.\n"
    "- Confirmatory CI/p values require all 5,000 locked replicates to be estimable for the metric.\n"
    "- Inputs: immutable Stage-05 prediction artifacts, admitted only through the canonical Stage-05 prediction/provenance root.\n"
)

required_hashes={x:sha256_file(OUT_ROOT/x) for x in required}
manifest={
    "schema_version":2,
    "stage_id":STAGE_ID,
    "protocol_version":PROTOCOL_VERSION,
    "upstream":{
        "mode":"DIRECT_CANONICAL_STAGE05_OUTPUT",
        "stage05_manifest_sha256":sha256_file(S5/"STAGE05_ARTIFACT_MANIFEST.json"),
        "stage05_checksum_sha256":sha256_file(S5/"STAGE05_SHA256SUMS.txt"),
        "stage05_status_sha256":sha256_file(S5/"STAGE_STATUS.json"),
        "stage05_identity":STAGE05_CANONICAL_IDENTITY,
        "prediction_sha256":physical_prediction_hashes,
        "h3_statistics_consumed":False,
    },
    "scientific_contract":{
        "budgets":BUDGETS,
        "primary_budget":PRIMARY_BUDGET,
        "bootstrap_reps":BOOTSTRAP_REPS,
        "bootstrap_seed":BOOTSTRAP_SEED,
        "seeds":SEEDS,
        "models":MODEL_KEYS,
        "positive_risk_and_uncertainty_review_separate":True,
        "conditional_valid_only_bootstrap_inference":False,
        "rejection_resampling":False,
    },
    "artifacts":required_hashes,
    "stage06_summary_sha256":sha256_file(OUT_ROOT/"STAGE06_SUMMARY.md"),
    "run_metadata_sha256":sha256_file(OUT_ROOT/"RUN_METADATA.json"),
}
manifest["manifest_payload_sha256"]=hashlib.sha256(cjb(manifest)).hexdigest()
write_json(OUT_ROOT/"REPRODUCIBILITY_MANIFEST.json",manifest)

entries=[]
for p in sorted(OUT_ROOT.rglob("*"),key=str):
    if p.is_file() and p.name not in {"STAGE06_SHA256SUMS.txt","STAGE_STATUS.json"}:
        rel=str(p.relative_to(OUT_ROOT)).replace("\\","/")
        h=required_hashes.get(rel)
        if h is None: h=sha256_file(p)
        entries.append((rel,h))
write_text(OUT_ROOT/"STAGE06_SHA256SUMS.txt","\n".join(f"{h}  {r}" for r,h in entries)+"\n")

set_status(
    "PASS",
    "H4 fixed-budget and selective-risk evaluation completed from canonical final Stage-05 predictions",
    {
        "output_root":str(OUT_ROOT),
        "checksum_entries":len(entries),
        "bootstrap_total_seconds":bootstrap_total_seconds,
        "stage05_manifest_sha256":sha256_file(S5/"STAGE05_ARTIFACT_MANIFEST.json"),
    },
)
print("STAGE 06 COMPLETE — PASS")


STAGE 06 STATUS: PASS — H4 fixed-budget and selective-risk evaluation completed from canonical final Stage-05 predictions
STAGE 06 COMPLETE — PASS
